# Phase 10 — Numpy vs Keras, le vrai coût

## Objectif

Mesurer ce que Keras fait gagner en pratique :
- temps d’entraînement ;
- coût par epoch ;
- facteur d’accélération ;
- qualité finale du modèle.

Partie 1 :
- dataset Breast Cancer

Partie 2 :
- dataset synthétique plus grand avec `make_classification`

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## 1. Fonctions utilitaires numpy

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

def bce_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [4]:
def run_numpy(X_tr, y_tr, X_te, y_te, epochs):
    np.random.seed(42)

    n_features = X_tr.shape[1]

    W1 = np.random.randn(n_features, 16) * np.sqrt(2 / n_features)
    b1 = np.zeros((1, 16))

    W2 = np.random.randn(16, 8) * np.sqrt(2 / 16)
    b2 = np.zeros((1, 8))

    W3 = np.random.randn(8, 1) * np.sqrt(2 / 8)
    b3 = np.zeros((1, 1))

    lr = 0.2
    losses = []

    start = time.time()

    for epoch in range(epochs):
        z1 = X_tr @ W1 + b1
        a1 = relu(z1)

        z2 = a1 @ W2 + b2
        a2 = relu(z2)

        z3 = a2 @ W3 + b3
        y_pred = sigmoid(z3).flatten()

        loss = bce_loss(y_tr, y_pred)
        losses.append(loss)

        n = len(X_tr)

        err3 = (y_pred - y_tr).reshape(-1, 1)
        dW3 = (a2.T @ err3) / n
        db3 = np.sum(err3, axis=0, keepdims=True) / n

        err2 = (err3 @ W3.T) * relu_grad(z2)
        dW2 = (a1.T @ err2) / n
        db2 = np.sum(err2, axis=0, keepdims=True) / n

        err1 = (err2 @ W2.T) * relu_grad(z1)
        dW1 = (X_tr.T @ err1) / n
        db1 = np.sum(err1, axis=0, keepdims=True) / n

        W3 -= lr * dW3
        b3 -= lr * db3
        W2 -= lr * dW2
        b2 -= lr * db2
        W1 -= lr * dW1
        b1 -= lr * db1

    train_time = time.time() - start

    a1_te = relu(X_te @ W1 + b1)
    a2_te = relu(a1_te @ W2 + b2)
    y_te_pred = sigmoid(a2_te @ W3 + b3).flatten()

    test_acc = np.mean((y_te_pred > 0.5) == y_te)
    final_loss = losses[-1]

    return train_time, final_loss, test_acc

In [5]:
def run_keras(X_tr, y_tr, X_te, y_te, epochs):
    tf.random.set_seed(42)

    model = keras.Sequential([
        keras.layers.Input(shape=(X_tr.shape[1],)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(8, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    start = time.time()

    history = model.fit(
        X_tr,
        y_tr,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )

    train_time = time.time() - start
    test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)

    return train_time, test_loss, test_acc, history.history

## 2. Partie 1 — Breast Cancer

In [6]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

EPOCHS = 200

numpy_time_bc, numpy_loss_bc, numpy_acc_bc = run_numpy(X_train, y_train, X_test, y_test, EPOCHS)
keras_time_bc, keras_loss_bc, keras_acc_bc, keras_hist_bc = run_keras(X_train, y_train, X_test, y_test, EPOCHS)

speedup_bc = numpy_time_bc / keras_time_bc

print("=== BREAST CANCER ===")
print(f"Numpy | time={numpy_time_bc:.2f}s | loss={numpy_loss_bc:.4f} | acc={numpy_acc_bc:.4f}")
print(f"Keras | time={keras_time_bc:.2f}s | loss={keras_loss_bc:.4f} | acc={keras_acc_bc:.4f}")
print(f"Facteur d'accélération Keras : x{speedup_bc:.2f}")

=== BREAST CANCER ===
Numpy | time=0.16s | loss=0.0274 | acc=0.9561
Keras | time=27.17s | loss=0.1337 | acc=0.9649
Facteur d'accélération Keras : x0.01


## 3. Partie 2 — Dataset synthétique plus grand

In [7]:
X_big, y_big = make_classification(
    n_samples=20000,
    n_features=30,
    n_informative=20,
    n_redundant=5,
    random_state=42
)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_big, y_big, test_size=0.2, random_state=42, stratify=y_big
)

scaler_big = StandardScaler()
Xb_train = scaler_big.fit_transform(Xb_train)
Xb_test = scaler_big.transform(Xb_test)

EPOCHS_BIG = 50

numpy_time_big, numpy_loss_big, numpy_acc_big = run_numpy(Xb_train, yb_train, Xb_test, yb_test, EPOCHS_BIG)
keras_time_big, keras_loss_big, keras_acc_big, keras_hist_big = run_keras(Xb_train, yb_train, Xb_test, yb_test, EPOCHS_BIG)

speedup_big = numpy_time_big / keras_time_big

print("=== DATASET SYNTHÉTIQUE ===")
print(f"Numpy | time={numpy_time_big:.2f}s | loss={numpy_loss_big:.4f} | acc={numpy_acc_big:.4f}")
print(f"Keras | time={keras_time_big:.2f}s | loss={keras_loss_big:.4f} | acc={keras_acc_big:.4f}")
print(f"Facteur d'accélération Keras : x{speedup_big:.2f}")

=== DATASET SYNTHÉTIQUE ===
Numpy | time=0.81s | loss=0.4800 | acc=0.7895
Keras | time=50.81s | loss=0.0857 | acc=0.9735
Facteur d'accélération Keras : x0.02


In [8]:
labels = ["Breast Cancer", "Synthétique"]
numpy_times = [numpy_time_bc, numpy_time_big]
keras_times = [keras_time_bc, keras_time_big]
speedups = [speedup_bc, speedup_big]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(labels))
w = 0.35

axes[0].bar(x - w/2, numpy_times, width=w, label="Numpy")
axes[0].bar(x + w/2, keras_times, width=w, label="Keras")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel("Temps (s)")
axes[0].set_title("Temps d'entraînement")
axes[0].legend()

axes[1].bar(labels, speedups)
axes[1].set_ylabel("Facteur d'accélération")
axes[1].set_title("Gain de vitesse de Keras")

for i, v in enumerate(speedups):
    axes[1].text(i, v + 0.05, f"x{v:.2f}", ha="center")

plt.tight_layout()
plt.savefig("phase10_timing_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

print("Figure sauvegardée : phase10_timing_comparison.png")

C:\Users\serge\AppData\Local\Temp\ipykernel_20560\1198876423.py:26: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Figure sauvegardée : phase10_timing_comparison.png


C:\Users\serge\AppData\Local\Temp\ipykernel_20560\1198876423.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Analyse attendue

Le but n’est pas seulement de dire que Keras est plus pratique.

Il faut montrer :
- combien de secondes il fait gagner ;
- si l’écart grandit avec la taille du dataset ;
- si la qualité prédictive reste comparable ou meilleure.

Conclusion attendue :
- Keras réduit fortement le coût d’implémentation ;
- Keras accélère aussi l’entraînement ;
- l’écart devient plus visible quand la taille des données augmente.